In [11]:
import mne
from mne.decoding import CSP
import numpy as np
import random
import matplotlib.pyplot as plt
import pandas as pd
import json

from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from mne.filter import filter_data
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [12]:
data_train = np.load("datos_procesados_triple/P2_post_training.mat.npz")
data_test = np.load("datos_procesados_triple/P2_post_test.mat.npz")

X_train = data_train["X"]
y_train = data_train["y"]

X_test = data_test["X"]
y_test = data_test["y"]

print("Shape train:", X_train.shape, y_train.shape)
print("Shape test:", X_test.shape, y_test.shape)

Shape train: (320, 512, 16) (320,)
Shape test: (320, 512, 16) (320,)


In [13]:
X_train = np.transpose(X_train, (0, 2, 1))
X_test = np.transpose(X_test, (0, 2, 1))

bands = [
    (8, 12),
    (12, 16),
    (16, 20),
    (20, 24),
    (24, 30)
]
fs=256
n_channels_csp=4

def extract_fbcsp_features(X_train, X_test, y_train, bands, fs, n_channels_csp):
    X_train_features = []
    X_test_features = []

    for (l_freq, h_freq) in bands:
        print(f"Procesando banda {l_freq}-{h_freq} Hz")

        # Filtrar
        X_train_f = filter_data(X_train, fs, l_freq, h_freq)
        X_test_f = filter_data(X_test, fs, l_freq, h_freq)

        # CSP por banda
        csp = CSP(
            n_components=n_channels_csp,
            reg='ledoit_wolf',
            log=True
        )

        X_train_csp = csp.fit_transform(X_train_f, y_train)
        X_test_csp = csp.transform(X_test_f)

        X_train_features.append(X_train_csp)
        X_test_features.append(X_test_csp)

    # Concatenar features de todas las bandas
    X_train_final = np.concatenate(X_train_features, axis=1)
    X_test_final = np.concatenate(X_test_features, axis=1)

    return X_train_final, X_test_final

In [14]:
def balance_clases(x_train, y_train):
    classes, counts = np.unique(y_train, return_counts=True)
    min_count = counts.min()

    indices = []

    for c in classes:
        class_idx = np.where(y_train == c)[0]
        sampled_idx = np.random.choice(class_idx, min_count, replace=False)
        indices.extend(sampled_idx)

    indices = np.array(indices)

    x_bal = x_train[indices]
    y_bal = y_train[indices]

    shuffle_idx = np.random.permutation(len(y_bal))

    x_train_balanced = x_bal[shuffle_idx]
    y_train_balanced = y_bal[shuffle_idx]
    return  x_train_balanced, y_train_balanced

In [15]:
X_train, y_train = balance_clases(X_train, y_train)
X_test, y_test = balance_clases(X_test, y_test)

X_train_fbcsp, X_test_fbcsp = extract_fbcsp_features(
    X_train, X_test, y_train, bands, fs, n_channels_csp
)

print("Shape features:", X_train_fbcsp.shape)

# -------------------------
# CLASIFICADOR FINAL
# -------------------------
clf = LinearDiscriminantAnalysis()

clf.fit(X_train_fbcsp, y_train)

y_pred = clf.predict(X_test_fbcsp)


acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy FBCSP: {acc:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Procesando banda 8-12 Hz
Setting up band-pass filter from 8 - 12 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 12.00 Hz
- Upper transition bandwidth: 3.00 Hz (-6 dB cutoff frequency: 13.50 Hz)
- Filter length: 423 samples (1.652 s)

Setting up band-pass filter from 8 - 12 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 12.00 Hz
- Upper transition bandwidth: 3.00 Hz (-6 dB cutoff fr